# Simulation Design: Controlled Experiments with Covariates

When benchmarking analysis methods or training ML models, you often need **controlled experiments** with known sources of variation. PointillSim's design module lets you:

1. Define **covariates** that vary across samples
2. Control which **effects** are active
3. Generate complete **experimental designs**
4. Track the **design matrix** for downstream analysis

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    DistanceBasedRule,
)

# Import design module
from pointillsim.design import (
    Covariate,
    CovariateSystem,
    EffectController,
    SimulationDesign,
    DesignMatrix,
    run_simulation_design,
)

# Import covariate-based rules
from pointillsim.rules import (
    CovariateBasedRule,
    ThresholdCovariateRule,
    SpatialCovariateRule,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.style.use('seaborn-v0_8-whitegrid')

---
## 1. Defining Covariates

A **covariate** is any variable that varies across samples and may influence the simulation. Covariates can be:

- **Continuous**: Age, distance from landmark, expression level
- **Categorical**: Treatment group, tissue region, batch ID

In [ ]:
# Create individual covariates

# Continuous covariate: disease severity (0-1 scale)
severity = Covariate(
    name='severity',
    covariate_type='continuous',
    range_min=0.0,
    range_max=1.0,
    description='Disease severity score (0=healthy, 1=severe)'
)

# Categorical covariate: treatment group
treatment = Covariate(
    name='treatment',
    covariate_type='categorical',
    categories=['control', 'drug_A', 'drug_B'],
    description='Treatment condition'
)

# Continuous covariate: cell density
density = Covariate(
    name='cell_density',
    covariate_type='continuous',
    range_min=0.5,
    range_max=2.0,
    description='Relative cell density multiplier'
)

print("Defined covariates:")
for cov in [severity, treatment, density]:
    print(f"  {cov}")

In [ ]:
# Create a covariate system
cov_system = CovariateSystem()
cov_system.add_covariate(severity)
cov_system.add_covariate(treatment)
cov_system.add_covariate(density)

print(f"CovariateSystem with {len(cov_system.covariates)} covariates:")
for name, cov in cov_system.covariates.items():
    print(f"  {name}: {cov.covariate_type}")

In [ ]:
# Sample from the covariate system
n_samples = 10

samples = cov_system.sample(n_samples, seed=42)

print(f"Sampled {n_samples} covariate combinations:")
samples_df = pd.DataFrame(samples)
samples_df

---
## 2. Covariate-Based Cell Type Rules

Use covariates to control cell type distributions within FOVs.

In [ ]:
# Setup for demonstrations
n_cell_types = 4
frame_size = 500

tissue = TissueCellTypes()
tissue.generate_types_and_markers(
    n_genes=30,
    n_cell_types=n_cell_types,
)
tissue._cell_type_names = ['Healthy', 'Stressed', 'Immune', 'Fibroblast']

cell_props = CellTypesProperties(n_cell_types=n_cell_types)

In [ ]:
# ThresholdCovariateRule: Step-like change at threshold
# When severity > 0.5, switch from healthy (type 0) to stressed (type 1)

threshold_rule = ThresholdCovariateRule(
    n_cell_types=n_cell_types,
    covariate_name='severity',
    threshold=0.5,
    below_type=0,   # Healthy
    above_type=1,   # Stressed
    transition_width=0.1,  # Smooth transition around threshold
)

# Visualize the threshold rule
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

severities = [0.2, 0.4, 0.6, 0.8]

for ax, sev in zip(axes, severities):
    # Generate FOV with this severity
    np.random.seed(42)
    
    # Set covariate value
    threshold_rule.set_covariate_value(sev)
    
    fov_dist = FOVDistribution(
        frame_size=frame_size,
        background_element=lambda: FrameWideElement(
            frame_size=frame_size,
            tipical_cell_spacing=15,
            rules=threshold_rule,
        ),
    )
    
    fov = fov_dist.generate_fov()
    
    ax.scatter(fov.cell_centroids[:, 0], fov.cell_centroids[:, 1],
               c=fov.class_instance, cmap='RdYlGn_r', vmin=0, vmax=n_cell_types-1,
               s=20, alpha=0.7)
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    
    # Count proportions
    counts = Counter(fov.class_instance)
    healthy_pct = 100 * counts.get(0, 0) / fov.n_cells
    stressed_pct = 100 * counts.get(1, 0) / fov.n_cells
    
    ax.set_title(f'Severity = {sev}\nHealthy: {healthy_pct:.0f}% | Stressed: {stressed_pct:.0f}%')

plt.suptitle('ThresholdCovariateRule: Disease Severity Affects Cell Composition', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# SpatialCovariateRule: Cell types vary with position
# Creates spatial gradients within FOVs

spatial_rule = SpatialCovariateRule(
    n_cell_types=n_cell_types,
    gradient_direction='horizontal',  # or 'vertical', 'radial', (dx, dy)
    type_at_low=0,   # Healthy on left
    type_at_high=2,  # Immune on right
)

# Visualize different gradient directions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

directions = ['horizontal', 'vertical', 'radial', (1, 1)]
titles = ['Horizontal', 'Vertical', 'Radial', 'Diagonal (1,1)']

for ax, direction, title in zip(axes, directions, titles):
    np.random.seed(42)
    
    rule = SpatialCovariateRule(
        n_cell_types=n_cell_types,
        gradient_direction=direction,
        type_at_low=0,
        type_at_high=2,
    )
    
    fov_dist = FOVDistribution(
        frame_size=frame_size,
        background_element=lambda r=rule: FrameWideElement(
            frame_size=frame_size,
            tipical_cell_spacing=12,
            rules=r,
        ),
    )
    
    fov = fov_dist.generate_fov()
    
    scatter = ax.scatter(fov.cell_centroids[:, 0], fov.cell_centroids[:, 1],
                         c=fov.class_instance, cmap='viridis', vmin=0, vmax=n_cell_types-1,
                         s=15, alpha=0.7)
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'{title} Gradient')

plt.suptitle('SpatialCovariateRule: Position-Dependent Cell Types', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

---
## 3. Effect Controller

The `EffectController` lets you toggle specific effects on/off and set their magnitudes.

In [ ]:
# Create an effect controller
controller = EffectController()

# Register available effects
controller.register_effect('batch_effects', enabled=True, magnitude=0.3)
controller.register_effect('vignetting', enabled=True, magnitude=0.2)
controller.register_effect('dropout', enabled=True, magnitude=0.15)
controller.register_effect('background_noise', enabled=False, magnitude=0.001)

print("EffectController status:")
for name, settings in controller.effects.items():
    status = '✓' if settings['enabled'] else '✗'
    print(f"  {status} {name}: magnitude={settings['magnitude']}")

In [ ]:
# Toggle effects for different experimental conditions

# Condition 1: Clean data (no effects)
controller.disable_all()
print("Condition 1 - Clean:")
print(f"  Active effects: {controller.get_active_effects()}")

# Condition 2: Realistic data (all effects)
controller.enable_all()
print("\nCondition 2 - Realistic:")
print(f"  Active effects: {controller.get_active_effects()}")

# Condition 3: Specific effects only
controller.disable_all()
controller.enable_effect('dropout')
controller.set_magnitude('dropout', 0.25)
print("\nCondition 3 - Dropout only (25%):")
print(f"  Active effects: {controller.get_active_effects()}")

---
## 4. Simulation Design

A `SimulationDesign` specifies the complete experimental setup.

In [ ]:
# Create a simulation design for a dose-response experiment

design = SimulationDesign(
    name='severity_dose_response',
    covariate_system=cov_system,
    n_replicates=3,  # 3 FOVs per condition
    covariate_grid={
        'severity': [0.0, 0.25, 0.5, 0.75, 1.0],  # 5 severity levels
        'treatment': ['control'],  # Single treatment for this design
        'cell_density': [1.0],  # Fixed density
    },
)

print(f"SimulationDesign: {design.name}")
print(f"  Conditions: {design.n_conditions}")
print(f"  Replicates per condition: {design.n_replicates}")
print(f"  Total FOVs: {design.total_fovs}")

In [ ]:
# View the design matrix
design_df = design.get_design_matrix()
print("Design Matrix:")
design_df

In [ ]:
# Create a more complex factorial design

factorial_design = SimulationDesign(
    name='treatment_x_severity',
    covariate_system=cov_system,
    n_replicates=2,
    covariate_grid={
        'severity': [0.0, 0.5, 1.0],  # 3 severity levels
        'treatment': ['control', 'drug_A', 'drug_B'],  # 3 treatments
        'cell_density': [1.0],
    },
)

print(f"Factorial Design: {factorial_design.name}")
print(f"  Conditions: 3 severity × 3 treatment = {factorial_design.n_conditions}")
print(f"  Total FOVs: {factorial_design.total_fovs}")

# View structure
factorial_df = factorial_design.get_design_matrix()
print("\nDesign structure:")
print(factorial_df.groupby(['severity', 'treatment']).size().unstack())

---
## 5. Running a Simulation Design

Execute the design to generate all FOVs with tracked metadata.

In [ ]:
# Define FOV generator that uses covariates

def create_fov_generator(covariate_values):
    """
    Create a FOV generator based on covariate values.
    
    The severity covariate controls the proportion of stressed cells.
    """
    severity_val = covariate_values.get('severity', 0.0)
    density_val = covariate_values.get('cell_density', 1.0)
    
    # Cell type proportions based on severity
    # Higher severity = more stressed cells (type 1), fewer healthy (type 0)
    healthy_prop = max(0.1, 0.7 - 0.6 * severity_val)
    stressed_prop = min(0.7, 0.1 + 0.6 * severity_val)
    immune_prop = 0.1 + 0.1 * severity_val  # Slight increase with severity
    fibro_prop = 1.0 - healthy_prop - stressed_prop - immune_prop
    
    props = [healthy_prop, stressed_prop, immune_prop, fibro_prop]
    props = [max(0, p) for p in props]  # Ensure non-negative
    total = sum(props)
    props = [p / total for p in props]  # Normalize
    
    # Cell spacing based on density
    cell_spacing = 15 / density_val
    
    fov_dist = FOVDistribution(
        frame_size=frame_size,
        background_element=lambda: FrameWideElement(
            frame_size=frame_size,
            tipical_cell_spacing=cell_spacing,
            rules=MixOfNCellTypesRule(
                n_cell_types=n_cell_types,
                list_N=list(range(n_cell_types)),
                proportions=props,
            ),
        ),
    )
    
    return fov_dist

print("FOV generator function defined")

In [ ]:
# Run the dose-response design
from tqdm import tqdm

results = []
design_matrix = design.get_design_matrix()

for idx, row in tqdm(design_matrix.iterrows(), total=len(design_matrix), desc="Generating FOVs"):
    covariate_values = {
        'severity': row['severity'],
        'treatment': row['treatment'],
        'cell_density': row['cell_density'],
    }
    
    # Create FOV generator for this condition
    fov_dist = create_fov_generator(covariate_values)
    
    # Generate FOV
    np.random.seed(int(row['condition_id'] * 1000 + row['replicate']))
    fov = fov_dist.generate_fov()
    cell_props.apply(fov)
    
    # Generate observations
    hybiss = HybISS_Setup(tissue)
    hybiss.observe_dots(fov)
    
    # Store results
    results.append({
        'condition_id': row['condition_id'],
        'replicate': row['replicate'],
        'severity': row['severity'],
        'treatment': row['treatment'],
        'n_cells': fov.n_cells,
        'n_dots': len(hybiss.make_pandas_df()),
        'fov': fov,
        'type_counts': Counter(fov.class_instance),
    })

results_df = pd.DataFrame(results)
print(f"\nGenerated {len(results_df)} FOVs")

In [ ]:
# Visualize the dose-response relationship
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Sample FOVs at different severity levels
severity_levels = [0.0, 0.5, 1.0]
for ax, sev in zip(axes[0], severity_levels):
    # Get first replicate at this severity
    fov_data = results_df[(results_df['severity'] == sev) & (results_df['replicate'] == 0)].iloc[0]
    fov = fov_data['fov']
    
    scatter = ax.scatter(fov.cell_centroids[:, 0], fov.cell_centroids[:, 1],
                         c=fov.class_instance, cmap='Set1', vmin=0, vmax=n_cell_types-1,
                         s=20, alpha=0.7)
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'Severity = {sev}\n({fov.n_cells} cells)')

# Row 2: Quantitative analysis
# Cell counts by severity
ax = axes[1, 0]
for sev in results_df['severity'].unique():
    data = results_df[results_df['severity'] == sev]
    ax.scatter([sev]*len(data), data['n_cells'], alpha=0.7, s=50)
means = results_df.groupby('severity')['n_cells'].mean()
ax.plot(means.index, means.values, 'r-', linewidth=2, label='Mean')
ax.set_xlabel('Severity')
ax.set_ylabel('Cell Count')
ax.set_title('Cell Count vs Severity')
ax.legend()

# Cell type proportions
ax = axes[1, 1]
type_names = ['Healthy', 'Stressed', 'Immune', 'Fibroblast']
colors = plt.cm.Set1(np.linspace(0, 1, n_cell_types))

for type_idx, (type_name, color) in enumerate(zip(type_names, colors)):
    props = []
    for sev in sorted(results_df['severity'].unique()):
        data = results_df[results_df['severity'] == sev]
        type_counts = [d['type_counts'].get(type_idx, 0) for d in data.to_dict('records')]
        total_counts = [d['n_cells'] for d in data.to_dict('records')]
        prop = np.mean([t/n if n > 0 else 0 for t, n in zip(type_counts, total_counts)])
        props.append(prop)
    ax.plot(sorted(results_df['severity'].unique()), props, '-o', 
            color=color, label=type_name, linewidth=2, markersize=8)

ax.set_xlabel('Severity')
ax.set_ylabel('Proportion')
ax.set_title('Cell Type Proportions vs Severity')
ax.legend()
ax.set_ylim(0, 1)

# Transcript counts
ax = axes[1, 2]
for sev in results_df['severity'].unique():
    data = results_df[results_df['severity'] == sev]
    ax.scatter([sev]*len(data), data['n_dots'], alpha=0.7, s=50)
means = results_df.groupby('severity')['n_dots'].mean()
ax.plot(means.index, means.values, 'r-', linewidth=2, label='Mean')
ax.set_xlabel('Severity')
ax.set_ylabel('Transcript Count')
ax.set_title('Transcript Count vs Severity')
ax.legend()

plt.suptitle('Dose-Response Design: Severity Affects Tissue Composition', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Design Matrix for Downstream Analysis

The `DesignMatrix` class tracks all covariate values and sample metadata.

In [ ]:
# Create a comprehensive design matrix from results
dm = DesignMatrix()

for _, row in results_df.iterrows():
    dm.add_sample(
        sample_id=f"sample_{int(row['condition_id'])}_{int(row['replicate'])}",
        covariates={
            'severity': row['severity'],
            'treatment': row['treatment'],
        },
        metadata={
            'n_cells': row['n_cells'],
            'n_dots': row['n_dots'],
        }
    )

print(f"DesignMatrix with {len(dm.samples)} samples")

In [ ]:
# Export design matrix for use in R or other tools
dm_df = dm.to_dataframe()
print("Design Matrix (exportable):")
dm_df

In [ ]:
# Filter samples by covariate values
high_severity = dm.filter_samples(severity=lambda x: x >= 0.75)
print(f"\nHigh severity samples (severity >= 0.75): {len(high_severity)}")

# Get unique conditions
conditions = dm.get_unique_conditions(['severity', 'treatment'])
print(f"\nUnique conditions: {len(conditions)}")
for cond in conditions[:5]:
    print(f"  {cond}")

---
## 7. Complete Example: Multi-Factor Experiment

Let's run a complete experiment varying multiple factors.

In [ ]:
# 3x3 factorial: Treatment × Severity
multi_design = SimulationDesign(
    name='full_factorial',
    covariate_system=cov_system,
    n_replicates=2,
    covariate_grid={
        'severity': [0.0, 0.5, 1.0],
        'treatment': ['control', 'drug_A', 'drug_B'],
        'cell_density': [1.0],
    },
)

print(f"Multi-factor design: {multi_design.name}")
print(f"  Total FOVs: {multi_design.total_fovs}")

In [ ]:
# Modified generator that responds to treatment
def create_fov_generator_treatment(covariate_values):
    """
    Create FOV generator that responds to both severity and treatment.
    
    - Control: natural severity response
    - Drug A: reduces stressed cell proportion
    - Drug B: increases immune cell proportion
    """
    severity_val = covariate_values.get('severity', 0.0)
    treatment = covariate_values.get('treatment', 'control')
    
    # Base proportions from severity
    healthy_prop = max(0.1, 0.7 - 0.6 * severity_val)
    stressed_prop = min(0.7, 0.1 + 0.6 * severity_val)
    immune_prop = 0.1 + 0.1 * severity_val
    fibro_prop = 0.1
    
    # Apply treatment effects
    if treatment == 'drug_A':
        # Drug A reduces stressed cells, increases healthy
        reduction = 0.5 * stressed_prop
        stressed_prop -= reduction
        healthy_prop += reduction
    elif treatment == 'drug_B':
        # Drug B increases immune response
        immune_prop += 0.15
    
    # Normalize
    props = [healthy_prop, stressed_prop, immune_prop, fibro_prop]
    props = [max(0, p) for p in props]
    total = sum(props)
    props = [p / total for p in props]
    
    fov_dist = FOVDistribution(
        frame_size=frame_size,
        background_element=lambda: FrameWideElement(
            frame_size=frame_size,
            tipical_cell_spacing=15,
            rules=MixOfNCellTypesRule(
                n_cell_types=n_cell_types,
                list_N=list(range(n_cell_types)),
                proportions=props,
            ),
        ),
    )
    
    return fov_dist

# Run multi-factor experiment
multi_results = []
multi_dm = multi_design.get_design_matrix()

for idx, row in tqdm(multi_dm.iterrows(), total=len(multi_dm), desc="Generating"):
    covariate_values = {
        'severity': row['severity'],
        'treatment': row['treatment'],
        'cell_density': row['cell_density'],
    }
    
    fov_dist = create_fov_generator_treatment(covariate_values)
    
    np.random.seed(int(row['condition_id'] * 1000 + row['replicate']))
    fov = fov_dist.generate_fov()
    
    multi_results.append({
        'severity': row['severity'],
        'treatment': row['treatment'],
        'replicate': row['replicate'],
        'n_cells': fov.n_cells,
        'healthy_prop': (fov.class_instance == 0).sum() / fov.n_cells,
        'stressed_prop': (fov.class_instance == 1).sum() / fov.n_cells,
        'immune_prop': (fov.class_instance == 2).sum() / fov.n_cells,
        'fibro_prop': (fov.class_instance == 3).sum() / fov.n_cells,
    })

multi_df = pd.DataFrame(multi_results)
print(f"\nCompleted: {len(multi_df)} FOVs")

In [ ]:
# Visualize treatment effects
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

treatments = ['control', 'drug_A', 'drug_B']
colors = {'control': 'gray', 'drug_A': 'blue', 'drug_B': 'green'}

# Stressed cells
ax = axes[0]
for treat in treatments:
    data = multi_df[multi_df['treatment'] == treat]
    means = data.groupby('severity')['stressed_prop'].mean()
    stds = data.groupby('severity')['stressed_prop'].std()
    ax.errorbar(means.index, means.values, yerr=stds.values, 
                fmt='-o', color=colors[treat], label=treat, 
                linewidth=2, markersize=8, capsize=5)
ax.set_xlabel('Severity')
ax.set_ylabel('Proportion')
ax.set_title('Stressed Cell Proportion\n(Drug A reduces stress response)')
ax.legend()
ax.set_ylim(0, 0.8)

# Healthy cells
ax = axes[1]
for treat in treatments:
    data = multi_df[multi_df['treatment'] == treat]
    means = data.groupby('severity')['healthy_prop'].mean()
    stds = data.groupby('severity')['healthy_prop'].std()
    ax.errorbar(means.index, means.values, yerr=stds.values,
                fmt='-o', color=colors[treat], label=treat,
                linewidth=2, markersize=8, capsize=5)
ax.set_xlabel('Severity')
ax.set_ylabel('Proportion')
ax.set_title('Healthy Cell Proportion\n(Drug A preserves healthy cells)')
ax.legend()
ax.set_ylim(0, 0.8)

# Immune cells
ax = axes[2]
for treat in treatments:
    data = multi_df[multi_df['treatment'] == treat]
    means = data.groupby('severity')['immune_prop'].mean()
    stds = data.groupby('severity')['immune_prop'].std()
    ax.errorbar(means.index, means.values, yerr=stds.values,
                fmt='-o', color=colors[treat], label=treat,
                linewidth=2, markersize=8, capsize=5)
ax.set_xlabel('Severity')
ax.set_ylabel('Proportion')
ax.set_title('Immune Cell Proportion\n(Drug B enhances immune response)')
ax.legend()
ax.set_ylim(0, 0.5)

plt.suptitle('Multi-Factor Experiment: Treatment × Severity Interaction', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

---
## Summary

This notebook covered PointillSim's design module for controlled experiments:

| Component | Purpose |
|-----------|--------|
| `Covariate` | Define continuous/categorical variables |
| `CovariateSystem` | Manage multiple covariates |
| `EffectController` | Toggle technical effects on/off |
| `SimulationDesign` | Specify experimental design |
| `DesignMatrix` | Track sample metadata |
| `CovariateBasedRule` | Cell types respond to covariates |

### Key Takeaways

1. **Covariates** let you systematically vary biological conditions
2. **Effect controllers** let you isolate specific sources of variation
3. **Factorial designs** enable studying interactions between factors
4. **Design matrices** facilitate downstream statistical analysis

### Next Steps

- **09_validation_and_difficulty.ipynb**: Assess simulation difficulty and validate results
- **10_advanced_structures.ipynb**: Explore advanced histological elements